# Star-domain vs star-domain interactions

The existing exclusion term in `radially_convex` works **circle-by-circle**: the boundary of set $A$ is pushed back from any circle belonging to set $B$. ...

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import SVG
from jax import numpy as jnp

from vizopt.animation import SnapshotCallback, snapshots_to_animated_svg
from vizopt.base import ObjectiveTerm, OptimConfig, OptimizationProblemTemplate
from vizopt.radially_convex import (
    _build_membership,
    _init_centers_and_radii,
    _multi_term_area,
    _multi_term_enclosure,
    _multi_term_min_radius,
    _multi_term_perimeter,
    _multi_term_smoothness,
    _svg_configuration_fixed,
)


## Random Fourier star domains

In [ ]:
def get_random_radii(thetas, seed):
    rng = np.random.default_rng(seed)
    k = 5
    random_fourier_coeffs = rng.random((k, 2))
    

    mult_thetas = thetas.reshape(-1, 1) * np.arange(1, k + 1)
    logradii = (np.cos(mult_thetas) * random_fourier_coeffs[:, 0]).sum(axis=1) + (
        np.sin(mult_thetas) * random_fourier_coeffs[:, 1]
    ).sum(axis=1)

    radii = logradii + 1 - min(0, min(logradii))
    return radii

thetas = np.linspace(0, 2 * np.pi, 100)
for seed in range(3):
    
    radii = get_random_radii(thetas, seed)

    x = radii * np.cos(thetas)
    y = radii * np.sin(thetas)

    _, ax = plt.subplots(figsize=(5, 5))
    ax.plot(x, y)


### Example: two star domains

In [ ]:
radii_a = get_random_radii(thetas, 0)
radii_b = get_random_radii(thetas, 1)

center_a = (0, 0)
center_b = (6, 6.7)

x_a = center_a[0] + radii_a * np.cos(thetas)
y_a = center_a[1] + radii_a * np.sin(thetas)

x_b = center_b[0] + radii_b * np.cos(thetas)
y_b = center_b[1] + radii_b * np.sin(thetas)

_, ax = plt.subplots(figsize=(5, 5))
ax.scatter(center_a[0], center_a[1])
ax.plot(x_a, y_a)

ax.scatter(center_b[0], center_b[1])
ax.plot(x_b, y_b)
plt.axis("equal")

In [ ]:
tan_theta_a_by_theta_b = (y_b - center_a[1]) / (x_b - center_a[0])
theta_a_by_theta_b = np.arctan(tan_theta_a_by_theta_b)
_, ax = plt.subplots(figsize=(4, 2))
ax.plot(180/np.pi * theta_a_by_theta_b)

There is a collision if and only if min(b_from_center_a - radii_a_by_theta_b) < 0

In [ ]:
_, ax = plt.subplots(figsize=(4, 3))
radii_a_by_theta_b = np.interp(theta_a_by_theta_b, thetas, radii_a)
plt.plot(radii_a_by_theta_b)

b_from_center_a = np.linalg.norm(np.array([x_b - center_a[0], y_b - center_a[1]]).T, axis=1)

plt.plot(b_from_center_a)

## Star-vs-star exclusion term

The current `_multi_term_exclusion` is **star vs circle**: for each ray of set $A$, it checks whether the boundary extends past the near face of any non-member circle. This means set $A$ can still enter the *interior* of set $B$ in gaps between $B$'s circles.

The **star-vs-star** approach instead asks: for each boundary point $p_k$ of set $A$, is $p_k$ inside the star domain of set $B$? This is computed by:

1. Finding the vector $d = p_k - c_B$ (direction and distance from $B$'s center)
2. Interpolating $B$'s radius at the angle of $d$: $R_B(\alpha)$
3. Penalising when $\|d\| < R_B(\alpha)$, i.e. $p_k$ is inside $B$

This is a strictly geometric check against the *actual optimised boundary* of $B$, not the underlying circles.

In [ ]:
def _multi_term_star_exclusion(optim_vars, input_params):
    """Star-vs-star exclusion: boundary of each set must not enter any other set's interior.

    For each ordered pair (s, t) with s != t and for each boundary point k of set s,
    penalises when that point lies inside the star domain of t — i.e. when its distance
    from center_t is less than t's radius interpolated at that angle.

    optim_vars keys: "centers" (S, 2), "radii" (S, K)
    input_params keys: "angles" (K,)
    """
    centers = optim_vars["centers"]  # (S, 2)
    radii = optim_vars["radii"]      # (S, K)
    angles = input_params["angles"]  # (K,)
    S, K = radii.shape

    # Boundary points for every set: (S, K, 2)
    directions = jnp.stack([jnp.cos(angles), jnp.sin(angles)], axis=-1)  # (K, 2)
    points = centers[:, None, :] + radii[:, :, None] * directions[None, :, :]  # (S, K, 2)

    # Vector from center_t to each boundary point of s:
    #   diff[s, t, k] = points[s, k] - centers[t]
    diff = points[:, None, :, :] - centers[None, :, None, :]  # (S, S, K, 2)

    # Distance from center_t to each boundary point of s
    dist = jnp.sqrt(jnp.sum(diff**2, axis=-1) + 1e-12)  # (S, S, K)

    # Angle from center_t to each boundary point of s  (in (-π, π])
    alpha = jnp.arctan2(diff[..., 1], diff[..., 0])  # (S, S, K)

    # Convert angle to fractional index in [0, K) and linearly interpolate t's radii
    delta_theta = 2 * jnp.pi / K
    frac_idx = (alpha % (2 * jnp.pi)) / delta_theta  # (S, S, K)

    idx_lo = jnp.floor(frac_idx).astype(jnp.int32) % K  # (S, S, K)
    idx_hi = (idx_lo + 1) % K                            # (S, S, K)
    w_hi = frac_idx - jnp.floor(frac_idx)                # (S, S, K)

    # r_lo[s, t, k] = radii[t, idx_lo[s, t, k]]
    t_range = jnp.arange(S)                                      # (S,)
    r_lo = radii[t_range[None, :, None], idx_lo]  # (S, S, K)
    r_hi = radii[t_range[None, :, None], idx_hi]  # (S, S, K)
    r_interp = (1.0 - w_hi) * r_lo + w_hi * r_hi  # (S, S, K)

    # Penalise when boundary point of s is inside t  (dist < r_interp)
    not_same = (jnp.arange(S)[:, None] != jnp.arange(S)[None, :])[:, :, None]  # (S, S, 1)
    violations = jnp.where(not_same, jnp.maximum(0.0, r_interp - dist), 0.0)
    return jnp.sum(violations**2)

In [ ]:
def _multi_term_star_enclosure(optim_vars, input_params):
    """Star-vs-star enclosure: boundary of inner sets must stay inside outer sets.

    For each (inner, outer) pair indicated by input_params["enclosure_mask"],
    penalises boundary points of the inner set that lie outside the star domain
    of the outer set — i.e. when their distance from center_outer exceeds the
    outer set's radius interpolated at that angle.

    optim_vars keys: "centers" (S, 2), "radii" (S, K)
    input_params keys: "angles" (K,), "enclosure_mask" (S, S) bool
      enclosure_mask[inner, outer] = True  →  inner must be inside outer
    """
    centers = optim_vars["centers"]        # (S, 2)
    radii   = optim_vars["radii"]          # (S, K)
    angles  = input_params["angles"]       # (K,)
    mask    = input_params["enclosure_mask"]  # (S, S) bool

    S, K = radii.shape

    # Boundary points of every set: (S, K, 2)
    directions = jnp.stack([jnp.cos(angles), jnp.sin(angles)], axis=-1)  # (K, 2)
    points = centers[:, None, :] + radii[:, :, None] * directions[None, :, :]  # (S, K, 2)

    # diff[inner, outer, k] = points[inner, k] - centers[outer]
    diff = points[:, None, :, :] - centers[None, :, None, :]  # (S, S, K, 2)

    # Distance from center_outer to each boundary point of inner
    dist  = jnp.sqrt(jnp.sum(diff**2, axis=-1) + 1e-12)  # (S, S, K)

    # Angle from center_outer to each boundary point of inner
    alpha = jnp.arctan2(diff[..., 1], diff[..., 0])        # (S, S, K)

    # Linearly interpolate outer's radius at that angle
    delta_theta = 2 * jnp.pi / K
    frac_idx = (alpha % (2 * jnp.pi)) / delta_theta        # (S, S, K) in [0, K)

    idx_lo = jnp.floor(frac_idx).astype(jnp.int32) % K    # (S, S, K)
    idx_hi = (idx_lo + 1) % K
    w_hi   = frac_idx - jnp.floor(frac_idx)

    # r_lo[inner, outer, k] = radii[outer, idx_lo[inner, outer, k]]
    outer_range = jnp.arange(S)                                        # (S,)
    r_lo = radii[outer_range[None, :, None], idx_lo]  # (S, S, K)
    r_hi = radii[outer_range[None, :, None], idx_hi]
    r_interp = (1.0 - w_hi) * r_lo + w_hi * r_hi      # (S, S, K)

    # Violation: boundary point of inner is OUTSIDE outer  →  dist > r_interp
    violations = jnp.where(
        mask[:, :, None],
        jnp.maximum(0.0, dist - r_interp),
        0.0,
    )
    return jnp.sum(violations**2)

In [ ]:
def optimize_star_vs_star(
    circles,
    sets,
    k_angles=32,
    weight_area=1.0,
    weight_perimeter=1.0,
    weight_exclusion=10.0,
    weight_enclosure=10.0,
    weight_smoothness=1.0,
    offsets=0.1,
    enclosures=None,
    optim_config=None,
    callback=None,
):
    """Like optimize_multiple_radially_convex_sets, but with star-vs-star terms.

    Swaps the circle-based exclusion for a star-vs-star exclusion, and adds an
    optional star-vs-star enclosure term for nested-set constraints.

    Args:
        circles: (N, 3) array [cx, cy, r].
        sets: list of S index subsets into circles.
        k_angles: number of angular samples per boundary.
        weight_area, weight_perimeter, weight_smoothness: loss weights.
        weight_exclusion: weight for the star-vs-star exclusion penalty
            (all pairs (s, t) with s ≠ t).
        weight_enclosure: weight for the star-vs-star enclosure penalty.
            Only active when `enclosures` is non-empty.
        offsets: padding added to circle radii in the enclosure term.
        enclosures: list of (inner_idx, outer_idx) pairs. Each pair means
            "the boundary of sets[inner_idx] must lie inside sets[outer_idx]."
            Default None (no enclosure constraint).
        optim_config: OptimConfig (uses defaults when None).
        callback: optional iteration callback.

    Returns:
        (results, history, problem) — same shape as optimize_multiple_radially_convex_sets.
    """
    circles_array = np.asarray(circles, dtype=np.float32)
    if circles_array.ndim == 1:
        circles_array = circles_array[None, :]
    N = len(circles_array)
    S = len(sets)
    angles = np.linspace(0, 2 * np.pi, k_angles, endpoint=False).astype(np.float32)

    membership = _build_membership(S, N, sets)
    initial_centers, initial_radii = _init_centers_and_radii(circles_array, sets, angles)
    offsets_array = np.broadcast_to(np.asarray(offsets, dtype=np.float32), (S, N)).copy()

    # Build enclosure mask (S, S): [inner, outer]
    enclosure_mask = np.zeros((S, S), dtype=bool)
    for inner, outer in (enclosures or []):
        enclosure_mask[inner, outer] = True

    input_parameters = {
        "circles": circles_array,
        "angles": angles,
        "membership": membership,
        "offsets": offsets_array,
        "enclosure_mask": enclosure_mask,
    }

    def initialize(_, seed):
        return {"centers": initial_centers, "radii": initial_radii}

    terms = [
        ObjectiveTerm("enclosure",     _multi_term_enclosure,      10.0),
        ObjectiveTerm("star_excl",     _multi_term_star_exclusion,  weight_exclusion),
        ObjectiveTerm("star_enclose",  _multi_term_star_enclosure,  weight_enclosure),
        ObjectiveTerm("min_radius",    _multi_term_min_radius,      10.0),
        ObjectiveTerm("smoothness",    _multi_term_smoothness,      weight_smoothness),
        ObjectiveTerm("area",          _multi_term_area,            weight_area),
        ObjectiveTerm("perimeter",     _multi_term_perimeter,       weight_perimeter),
    ]

    problem = OptimizationProblemTemplate(
        terms=terms,
        initialize=initialize,
        svg_configuration=_svg_configuration_fixed,
    ).instantiate(input_parameters)

    optim_vars, history = problem.optimize(optim_config, callback=callback)

    results = [
        {
            "center": np.array(optim_vars["centers"][s]),
            "radii":  np.array(optim_vars["radii"][s]),
            "angles": angles,
        }
        for s in range(S)
    ]
    return results, history, problem

## Demo: star-vs-star vs star-vs-circle exclusion

Two interleaved groups of circles where the star boundaries would naturally overlap.
The **star-vs-circle** approach (left) can let boundaries slip into the gap between
circles of the opposing set; **star-vs-star** (right) enforces separation against the
full optimised boundary.

In [ ]:
from vizopt.radially_convex import optimize_multiple_radially_convex_sets
from vizopt.base import OptimConfig

# Two groups of circles that sit close together so their envelopes want to overlap.
# Group A (indices 0-2): small cluster on the left
# Group B (indices 3-5): small cluster on the right, partially interleaved
circles = np.array([
    # Group A
    [-0.3,  1.3, 0.4],
    [-0.6, -0.3, 0.4],
    [0.5, -0.4, 0.35],
    # Group B
    [2.0,  0.5, 0.4],
    [2.7, -0.3, 0.4],
    [0.5, 0.7, 0.35],
], dtype=np.float32)

sets = [{0, 1, 2}, {3, 4, 5}]

config = OptimConfig(n_iters=3000, learning_rate=5e-3)

# --- star-vs-circle (existing) ---
results_svc, history_svc, problem_svc = optimize_multiple_radially_convex_sets(
    circles, sets, k_angles=64, weight_exclusion=20.0, offsets=0.15,
    optim_config=config,
)

# --- star-vs-star (new) ---
results_svs, history_svs, problem_svs = optimize_star_vs_star(
    circles, sets, k_angles=64, weight_exclusion=20.0, offsets=0.15,
    optim_config=config,
)

In [ ]:
from vizopt.utils import _SVG_SET_COLORS

def plot_results(ax, results, circles, title):
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")
    ax.axis("off")

    # Draw circles
    for i_circle, (cx, cy, r) in enumerate(circles):
        patch = plt.Circle((cx, cy), r, color="#4472c4", alpha=0.45, zorder=2)
        ax.add_patch(patch)
        ax.plot(cx, cy, "o", color="#2a52a0", ms=2, zorder=3)
        ax.text(cx, cy + r + 0.1, f"circle {i_circle}", ha="center", fontsize=8, color="#2a52a0")

    # Draw star boundaries
    for s, res in enumerate(results):
        color = _SVG_SET_COLORS[s % len(_SVG_SET_COLORS)]
        angs = res["angles"]
        radii = res["radii"]
        cx, cy = res["center"]
        bx = cx + radii * np.cos(angs)
        by = cy + radii * np.sin(angs)
        # close the polygon
        bx = np.append(bx, bx[0])
        by = np.append(by, by[0])
        ax.fill(bx, by, color=color, alpha=0.15, zorder=1)
        ax.plot(bx, by, color=color, lw=1.5, zorder=2)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_results(axes[0], results_svc, circles, "Star-vs-circle exclusion (existing)")
plot_results(axes[1], results_svs, circles, "Star-vs-star exclusion (new)")

for ax in axes:
    ax.autoscale_view()

plt.tight_layout()
plt.show()

In [ ]:
# Animated SVG of the star-vs-star run
cb = SnapshotCallback(every=50)
_, history_anim, problem_anim = optimize_star_vs_star(
    circles, sets, k_angles=64, weight_exclusion=20.0, offsets=0.15,
    optim_config=OptimConfig(n_iters=3000, learning_rate=5e-3),
    callback=cb,
)
svg = snapshots_to_animated_svg(problem_anim, cb.snapshots, fps=12, size=450,
                                 history=history_anim, log_scale=True)
SVG(data=svg)

## Demo: star-vs-star enclosure (nested sets)

Three sets in a containment hierarchy: set 0 ⊂ set 1 ⊂ set 2.
The `enclosures=[(0,1),(1,2)]` argument enforces those two nesting constraints simultaneously.

In [ ]:
# Three concentric groups of circles, one per nesting level.
# Set 0 (innermost) — tight cluster near the origin
# Set 1 (middle)    — broader ring around it
# Set 2 (outermost) — wide ring further out
circles_nested = np.array([
    # Set 0 — inner
    [ 0.0,  0.3, 0.25],
    [-0.3, -0.2, 0.25],
    [ 0.3, -0.2, 0.25],
    # Set 1 — middle
    [ 0.0,  1.4, 0.3],
    [-1.4,  0.0, 0.3],
    [ 1.4,  0.0, 0.3],
    [ 0.0, -1.4, 0.3],
    # Set 2 — outer
    [ 0.0,  2.8, 0.35],
    [-2.0,  2.0, 0.35],
    [-2.8,  0.0, 0.35],
    [-2.0, -2.0, 0.35],
    [ 0.0, -2.8, 0.35],
    [ 2.0, -2.0, 0.35],
    [ 2.8,  0.0, 0.35],
    [ 2.0,  2.0, 0.35],
], dtype=np.float32)

sets_nested = [
    {0, 1, 2},           # set 0 — inner
    {3, 4, 5, 6},        # set 1 — middle
    {7, 8, 9, 10, 11, 12, 13, 14},  # set 2 — outer
]

# set 0 must be inside set 1; set 1 must be inside set 2
enclosures_nested = [(0, 1), (1, 2)]

config_nested = OptimConfig(n_iters=4000, learning_rate=5e-3)

cb_nested = SnapshotCallback(every=50)
results_nested, history_nested, problem_nested = optimize_star_vs_star(
    circles_nested, sets_nested,
    k_angles=64,
    weight_exclusion=0.0,   # no exclusion — sets are nested, not disjoint
    weight_enclosure=30.0,
    weight_area=0.5,
    weight_perimeter=0.5,
    weight_smoothness=1.0,
    offsets=0.12,
    enclosures=enclosures_nested,
    optim_config=config_nested,
    callback=cb_nested,
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_results(ax, results_nested, circles_nested, "Star-vs-star enclosure  (set 0 ⊂ set 1 ⊂ set 2)")
ax.autoscale_view()
plt.tight_layout()
plt.show()

svg_nested = snapshots_to_animated_svg(
    problem_nested, cb_nested.snapshots, fps=12, size=450,
    history=history_nested, log_scale=True,
)
SVG(data=svg_nested)